<a href="https://colab.research.google.com/github/hamsikee/CoMM/blob/main/figure5_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Figure 5 — CoMM loss ablation on bimodal Trifeatures (Colab)

Reproduces **Figure 5** of *What to align in multimodal contrastive learning?* (ICLR 2025):
linear probing accuracy of redundancy **R**, uniqueness **U** and synergy **S** along training,
when each term of `L_CoMM = L + Σᵢ Lᵢ` is optimized separately.

| arm | objective | expected |
|---|---|---|
| `sum_Li` | `Σᵢ Lᵢ` | R, U learned — **S stays at chance** |
| `L_only` | `L = −Î(Z′,Z″)` | all three learned, but **slowly** |
| `comm`  | `L + Σᵢ Lᵢ` | all three learned quickly |

Two dataset settings are run: `biased=false` (Experiment 1 → R, U) and `biased=true`
(Experiment 2, texture↔color mapping M → S becomes learnable).

**Runtime** on a T4: ~1 min/epoch of compute plus checkpoint writes to Drive, so budget
**~1.4 min/epoch** → at `EPOCHS = 50` each of the 6 runs takes ~70 min, ~7 h in total.
Each arm gets **its own cell** so a dropped session costs at most one arm — and re-running that
same cell **resumes from the last checkpoint** (see §5).

## 1. Runtime check
Set **Runtime → Change runtime type → GPU** before running.

In [ ]:
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("!! No GPU — switch the runtime type, otherwise a single run takes days.")

## 2. Repository

`run_figure5.py`, `plot_figure5.py` and `TrifeaturesLinearProbingCallback` are local additions,
so cloning upstream would miss them. Upload **`CoMM.zip`** (built locally — code plus the
2 600 Trifeatures PNGs, without `.venv/` and the MultiBench `.pkl` files) to the root of your Drive.

It is unpacked to `/content`, the local SSD, because running from the Drive FUSE mount is much
slower. Only the **logs and checkpoints** stay on Drive.

In [ ]:
import os, sys
from google.colab import drive
drive.mount('/content/drive')

ZIP  = '/content/drive/MyDrive/CoMM.zip'   # <- upload this
REPO = '/content/CoMM'
OUT  = '/content/drive/MyDrive/fig5_logs'  # <- logs + checkpoints, survive a disconnect

if not os.path.isfile(os.path.join(REPO, 'run_figure5.py')):
    assert os.path.isfile(ZIP), f'{ZIP} not found — upload CoMM.zip to your Drive first'
    !unzip -q -o "$ZIP" -d /content/
assert os.path.isfile(os.path.join(REPO, 'run_figure5.py')), f'unexpected zip layout under {REPO}'

os.makedirs(OUT, exist_ok=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
print(os.getcwd())

In [ ]:
# Colab already ships torch / lightning / sklearn / tensorboard; only hydra is missing.
!pip install -q "hydra-core==1.3.2" "pytorch-lightning>=2.1" einops

## 3. Trifeatures images and shared settings

2 400 train + 200 test PNGs, shipped inside the zip. `DATA_ROOT` is passed to every run as
`+data.data_module.root=...`, so `dataset/catalog.json` (which holds a Windows path) is bypassed.
If the images are missing they are re-rendered here — a few minutes, once.

`COMMON` holds the flags shared by every run, so `EPOCHS` is changed in one place.

In [31]:
DATA_ROOT = os.path.join(REPO, 'data', 'trifeatures_3combi')
EPOCHS = 30    # the paper uses 100; 50 already separates the three arms clearly

if not os.path.isdir(os.path.join(DATA_ROOT, 'train')):
    from dataset.trifeatures import Trifeatures
    Trifeatures(DATA_ROOT, split='train')   # renders both splits on first call
    Trifeatures(DATA_ROOT, split='test')

for split, expected in [('train', 2400), ('test', 200)]:
    n = len(os.listdir(os.path.join(DATA_ROOT, split)))
    print(f'{split}: {n} images', '' if n == expected else f'(expected {expected}!)')

COMMON = f'--epochs {EPOCHS} --max-size 2500 --num-workers 2 --root "{DATA_ROOT}" --out "{OUT}" --check-val-every-n-epoch 2'
print(COMMON)

train: 2400 images 
test: 200 images 
--epochs 30 --max-size 2500 --num-workers 2 --root "/content/CoMM/data/trifeatures_3combi" --out "/content/drive/MyDrive/fig5_logs" --check-val-every-n-epoch 2


In [32]:
print(repr(COMMON))

'--epochs 30 --max-size 2500 --num-workers 2 --root "/content/CoMM/data/trifeatures_3combi" --out "/content/drive/MyDrive/fig5_logs" --check-val-every-n-epoch 2'


## 4. Wiring check (~5 min)

2 epochs on a small subset. Expect four `Linear probe (...)` lines per epoch.
Skip this once it has passed.

It writes to `$OUT` **on purpose** — timing a check against local disk while the real runs write
to Drive gives a misleadingly fast number.

In [ ]:
!python run_figure5.py --arms comm --biased false --epochs 2 --max-size 512 \
    --num-workers 2 --root "$DATA_ROOT" --out "$OUT"

## 5. Experiment 1 — `biased=false` (redundancy / uniqueness)

**One arm per cell, ~70 min each.** Run them in order; you may stop between cells.

Every run checkpoints into
`<OUT>/CoMM/bimodal_trifeatures/fig5_biased-*_<arm>_seed42/checkpoints/`
**every 10 epochs**, keeping only the latest file. So if a cell dies halfway:

* **just re-run the same cell** — it prints `=> resuming from ...` and continues from the last
  saved epoch (at most 10 epochs are redone);
* `--no-resume` forces a clean retrain; `--ckpt-every-n-epochs 0` skips checkpointing entirely.

**Why every 10 and not every epoch?** A checkpoint is ~120 MB (weights + optimizer state), and
writing that onto the Drive FUSE mount takes minutes — far longer than the training epoch itself.
Saving every epoch measured **4.5 min/epoch** against ~1 min/epoch of actual compute. Every 10
epochs amortises it to roughly **1.4 min/epoch** while bounding what a crash costs.
Only one checkpoint is kept per run, so disk use stays at ~120 MB × 6 ≈ 700 MB.

Keep `EPOCHS` a multiple of 10 — otherwise the final epochs land after the last checkpoint, and
re-running a *finished* cell would redo them instead of exiting straight away.

In [33]:
  !python run_figure5.py --biased false --arms comm $COMMON

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
                                                               loss: 0.308      
                                                               ssl_acc: 83.333  
                                                               ssl_acc_0:       
                                                               100.000          
                                                               ssl_acc_1: 50.000
                                                               ssl_acc_2:       
                                                               100.000          
                                                               ssl_loss_0: 0.208
                                                               ssl_loss_1: 0.655
                                                               ssl_loss_2: 0.061
Validation  ━━━━━━━━━━━━━━━━━ 40/40 0:00:55 • 0:00:00 0.77it/s                  [2026-07-28 13:21:53,693][Linear probing][INFO] - Test acc@1/acc@5/acc_per

In [ ]:
!python run_figure5.py --biased false --arms sum_Li $COMMON

In [ ]:
!python run_figure5.py --biased false --arms L_only $COMMON

## 6. Experiment 2 — `biased=true` (synergy)

Same three arms, but the training pairs additionally respect the texture↔color mapping M,
which is what makes synergy learnable. Best run in a **separate session** (~2.5 h).

In [ ]:
!python run_figure5.py --biased true --arms comm $COMMON

In [ ]:
!python run_figure5.py --biased true --arms sum_Li $COMMON

In [ ]:
!python run_figure5.py --biased true --arms L_only $COMMON

### Which runs are already done?

In [ ]:
import glob, sys
sys.path.insert(0, REPO)
from plot_figure5 import find_runs, read_curves

runs = find_runs(OUT)
for key in sorted(runs):
    biased, arm, seed = key
    n = len(read_curves(runs[key], ['acc1_share'])[0])
    ckpts = glob.glob(os.path.join(OUT, 'CoMM', 'bimodal_trifeatures',
                                   f'fig5_biased-{biased}_{arm}_seed{seed}', 'checkpoints', '*.ckpt'))
    last = os.path.basename(max(ckpts, key=os.path.getmtime)) if ckpts else '-'
    print(f"biased={biased:5s} {arm:7s} seed={seed}  epochs logged: {n:3d}  checkpoint: {last}")

### Knobs if the runs are too slow

An epoch is roughly: 75–80 % contrastive training, 20–25 % linear probing (the probe extracts
features once and reuses them for the four tasks), plus the amortised Drive checkpoint write.

| flag | saving | cost |
|---|---|---|
| lower `EPOCHS` | **proportional** | the ranking between arms is clear well before 100 |
| `--max-size 5000` | ~40 % | departs from the paper's 10 000 training pairs |
| `--out /content/fig5_logs` | ~25 % | **fastest**, but a recycled VM takes the logs with it — copy them to Drive after each arm |
| `--ckpt-every-n-epochs 0` | ~25 % | no checkpoints at all, so no resuming |
| `--check-val-every-n-epoch 2` | ~10 % | half the resolution on the curves |
| `--no-fastsearch` | *slower* | full cost sweep, exactly as in the paper |

For error bars, repeat with `--seed 1 … 5`; `plot_figure5.py` averages seeds and shades ±1 std.

## 7. Plot

Works on whatever runs exist so far — you do not have to wait for all six.

In [ ]:
!python plot_figure5.py --logs "$OUT" --split-biased

from IPython.display import Image, display
display(Image(os.path.join(OUT, 'figure5.png')))

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "$OUT"